# Province-Level Daily Rainfall Forecasting

This notebook trains one recent-weighted two-stage XGBoost forecasting pipeline per province and exports the next 12 months of daily forecasts into `modeling/result` using the same filenames expected by the dashboard.

In [1]:
from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import fbeta_score, log_loss, mean_absolute_error
from xgboost import XGBClassifier, XGBRegressor

warnings.filterwarnings("ignore")


In [2]:
LAG_STEPS = [1, 2, 3, 7, 12, 14, 21, 30, 60, 90, 365]
ROLLING_WINDOWS = [7, 12, 30]
RAIN_COUNT_WINDOWS = [7, 14, 30]
RAIN_THRESHOLD = 1.0
TRAIN_YEARS = 15
RANDOM_STATE = 42
THRESHOLD_GRID = [0.18, 0.22, 0.26, 0.30, 0.34, 0.38]


def resolve_modeling_dir() -> Path:
    cwd = Path.cwd()
    if (cwd / "data_splitted").exists() and (cwd / "result").exists():
        return cwd
    if (cwd / "modeling").exists():
        return cwd / "modeling"
    raise FileNotFoundError("Could not locate the modeling directory.")


MODELING_DIR = resolve_modeling_dir()
DATA_DIR = MODELING_DIR / "data_splitted"
RESULT_DIR = MODELING_DIR / "result"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

province_files = sorted(DATA_DIR.glob("*_rainfall_1981_2025.csv"))
if not province_files:
    raise FileNotFoundError(f"No province CSV files found in {DATA_DIR}")

print(f"Modeling directory: {MODELING_DIR}")
print(f"Province files found: {len(province_files)}")


Modeling directory: C:\Users\Administrator\Desktop\DataViz_FinalPrj\COMP4010-Project-2\modeling
Province files found: 13


## Feature Engineering

In [3]:
def add_calendar_features(frame: pd.DataFrame) -> pd.DataFrame:
    data = frame.copy()
    data["date"] = pd.to_datetime(data["date"])
    data["year"] = data["date"].dt.year
    data["month"] = data["date"].dt.month
    data["day"] = data["date"].dt.day
    data["day_of_year"] = data["date"].dt.dayofyear
    data["quarter"] = data["date"].dt.quarter
    data["season_4"] = ((data["month"] - 1) // 3) + 1
    data["is_rainy_season"] = data["month"].isin([5, 6, 7, 8, 9, 10, 11]).astype(int)
    data["month_sin"] = np.sin(2 * np.pi * data["month"] / 12)
    data["month_cos"] = np.cos(2 * np.pi * data["month"] / 12)
    data["day_of_year_sin"] = np.sin(2 * np.pi * data["day_of_year"] / 365.25)
    data["day_of_year_cos"] = np.cos(2 * np.pi * data["day_of_year"] / 365.25)
    return data


def add_state_features(frame: pd.DataFrame) -> pd.DataFrame:
    data = frame.copy().sort_values("date").reset_index(drop=True)
    data["rain_indicator"] = (data["rainfall_mm"] >= RAIN_THRESHOLD).astype(int)

    for lag in LAG_STEPS:
        data[f"lag_{lag}"] = data["rainfall_mm"].shift(lag)

    lag_base = data["rainfall_mm"].shift(1)
    for window in ROLLING_WINDOWS:
        roll = lag_base.rolling(window=window, min_periods=window)
        data[f"rolling_mean_{window}"] = roll.mean()
        data[f"rolling_std_{window}"] = roll.std()
        data[f"rolling_sum_{window}"] = roll.sum()
        data[f"rolling_max_{window}"] = roll.max()
        data[f"rolling_min_{window}"] = roll.min()

    rain_prev = data["rain_indicator"].shift(1)
    for window in RAIN_COUNT_WINDOWS:
        data[f"rolling_rain_count_{window}"] = rain_prev.rolling(window=window, min_periods=window).sum()

    days_since_last_rain = []
    wet_streak_prev = []
    dry_streak_prev = []
    last_rain_date = None
    wet_streak = 0
    dry_streak = 0

    for row in data.itertuples(index=False):
        if last_rain_date is None:
            days_since_last_rain.append(np.nan)
        else:
            days_since_last_rain.append((row.date - last_rain_date).days)

        wet_streak_prev.append(wet_streak)
        dry_streak_prev.append(dry_streak)

        if row.rainfall_mm >= RAIN_THRESHOLD:
            wet_streak += 1
            dry_streak = 0
            last_rain_date = row.date
        else:
            dry_streak += 1
            wet_streak = 0

    data["days_since_last_rain"] = days_since_last_rain
    data["wet_streak_prev"] = wet_streak_prev
    data["dry_streak_prev"] = dry_streak_prev
    return data


def build_training_frame(frame: pd.DataFrame):
    data = add_calendar_features(frame)
    data = add_state_features(data)
    data["rain_occurrence"] = (data["rainfall_mm"] >= RAIN_THRESHOLD).astype(int)
    data["target_sqrt_rainfall"] = np.sqrt(data["rainfall_mm"])

    feature_columns = [
        *[f"lag_{lag}" for lag in LAG_STEPS],
        *[f"rolling_mean_{window}" for window in ROLLING_WINDOWS],
        *[f"rolling_std_{window}" for window in ROLLING_WINDOWS],
        *[f"rolling_sum_{window}" for window in ROLLING_WINDOWS],
        *[f"rolling_max_{window}" for window in ROLLING_WINDOWS],
        *[f"rolling_min_{window}" for window in ROLLING_WINDOWS],
        *[f"rolling_rain_count_{window}" for window in RAIN_COUNT_WINDOWS],
        "month_sin",
        "month_cos",
        "day_of_year_sin",
        "day_of_year_cos",
        "quarter",
        "season_4",
        "is_rainy_season",
        "days_since_last_rain",
        "wet_streak_prev",
        "dry_streak_prev",
    ]

    model_data = data.dropna(subset=feature_columns + ["rainfall_mm"]).reset_index(drop=True)
    max_date = model_data["date"].max()
    train_start = max_date - pd.DateOffset(years=TRAIN_YEARS)
    model_data = model_data[model_data["date"] >= train_start].reset_index(drop=True)
    return model_data, feature_columns


## Sample Weights and Validation

In [4]:
def time_validation_split(model_data: pd.DataFrame):
    if len(model_data) < 900:
        return model_data, None

    val_size = min(365, max(180, len(model_data) // 8))
    split_idx = len(model_data) - val_size
    train_df = model_data.iloc[:split_idx].copy()
    val_df = model_data.iloc[split_idx:].copy()

    if train_df.empty or val_df.empty:
        return model_data, None
    return train_df, val_df


def recency_weights(frame: pd.DataFrame, min_weight: float = 1.0, max_weight: float = 3.0) -> np.ndarray:
    if len(frame) <= 1:
        return np.array([max_weight])
    order = np.linspace(0.0, 1.0, len(frame))
    return min_weight + (max_weight - min_weight) * order


def classifier_weights(frame: pd.DataFrame) -> np.ndarray:
    weights = recency_weights(frame)
    rain_flag = frame["rain_occurrence"].to_numpy(dtype=float)
    heavy_rain_flag = (frame["rainfall_mm"] >= 20).to_numpy(dtype=float)
    extreme_rain_flag = (frame["rainfall_mm"] >= 50).to_numpy(dtype=float)
    weights = weights + 0.8 * rain_flag + 0.6 * heavy_rain_flag + 0.8 * extreme_rain_flag
    return weights


def regressor_weights(frame: pd.DataFrame) -> np.ndarray:
    weights = recency_weights(frame, min_weight=1.0, max_weight=3.5)
    rainfall = frame["rainfall_mm"].to_numpy(dtype=float)
    weights = weights + 1.0 * (rainfall >= 10) + 2.0 * (rainfall >= 30) + 3.0 * (rainfall >= 50)
    return weights


## Model Selection

In [5]:
def build_classifier(params) -> XGBClassifier:
    return XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        **params,
    )


def build_regressor(params) -> XGBRegressor:
    return XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        **params,
    )


def choose_threshold(y_true, probabilities):
    best_threshold = THRESHOLD_GRID[0]
    best_score = -1.0
    for threshold in THRESHOLD_GRID:
        preds = (probabilities >= threshold).astype(int)
        score = fbeta_score(y_true, preds, beta=2, zero_division=0)
        if score > best_score:
            best_score = score
            best_threshold = threshold
    return best_threshold, best_score


def tune_classifier(train_df: pd.DataFrame, val_df: pd.DataFrame, feature_columns):
    candidate_params = [
        {
            "n_estimators": 500,
            "max_depth": 5,
            "learning_rate": 0.04,
            "min_child_weight": 4,
            "gamma": 0.10,
            "subsample": 0.88,
            "colsample_bytree": 0.88,
            "reg_alpha": 0.05,
            "reg_lambda": 1.2,
        },
        {
            "n_estimators": 700,
            "max_depth": 4,
            "learning_rate": 0.03,
            "min_child_weight": 5,
            "gamma": 0.15,
            "subsample": 0.82,
            "colsample_bytree": 0.82,
            "reg_alpha": 0.08,
            "reg_lambda": 1.5,
        },
    ]

    if val_df is None or val_df["rain_occurrence"].nunique() < 2:
        return candidate_params[0], THRESHOLD_GRID[2]

    best_key = None
    best_params = candidate_params[0]
    best_threshold = THRESHOLD_GRID[2]

    for params in candidate_params:
        model = build_classifier(params)
        model.fit(
            train_df[feature_columns],
            train_df["rain_occurrence"],
            sample_weight=classifier_weights(train_df),
        )
        probabilities = model.predict_proba(val_df[feature_columns])[:, 1]
        threshold, f2 = choose_threshold(val_df["rain_occurrence"], probabilities)
        clipped_probabilities = np.clip(probabilities, 1e-6, 1 - 1e-6)
        loss = log_loss(val_df["rain_occurrence"], clipped_probabilities)
        candidate_key = (round(f2, 6), round(-loss, 6))
        if best_key is None or candidate_key > best_key:
            best_key = candidate_key
            best_params = params
            best_threshold = threshold

    return best_params, best_threshold


def tune_regressor(train_df: pd.DataFrame, val_df: pd.DataFrame, feature_columns):
    candidate_params = [
        {
            "n_estimators": 500,
            "max_depth": 5,
            "learning_rate": 0.04,
            "min_child_weight": 4,
            "gamma": 0.05,
            "subsample": 0.88,
            "colsample_bytree": 0.88,
            "reg_alpha": 0.04,
            "reg_lambda": 1.2,
        },
        {
            "n_estimators": 700,
            "max_depth": 4,
            "learning_rate": 0.03,
            "min_child_weight": 5,
            "gamma": 0.10,
            "subsample": 0.82,
            "colsample_bytree": 0.82,
            "reg_alpha": 0.06,
            "reg_lambda": 1.4,
        },
    ]

    rainy_train = train_df[train_df["rain_occurrence"] == 1].copy()
    if rainy_train.empty:
        return candidate_params[0], 1.0

    if val_df is None:
        return candidate_params[0], 1.0

    rainy_val = val_df[val_df["rain_occurrence"] == 1].copy()
    if rainy_val.empty:
        return candidate_params[0], 1.0

    best_mae = None
    best_params = candidate_params[0]
    best_scale = 1.0

    for params in candidate_params:
        model = build_regressor(params)
        model.fit(
            rainy_train[feature_columns],
            rainy_train["target_sqrt_rainfall"],
            sample_weight=regressor_weights(rainy_train),
        )
        pred_sqrt = model.predict(rainy_val[feature_columns])
        pred_mm = np.square(np.clip(pred_sqrt, a_min=0, a_max=None))
        raw_mean = float(pred_mm.mean()) if len(pred_mm) else 0.0
        actual_mean = float(rainy_val["rainfall_mm"].mean())
        scale = 1.0 if raw_mean <= 0 else np.clip(actual_mean / raw_mean, 0.85, 1.60)
        pred_mm = pred_mm * scale
        mae = mean_absolute_error(rainy_val["rainfall_mm"], pred_mm)
        if best_mae is None or mae < best_mae:
            best_mae = mae
            best_params = params
            best_scale = scale

    return best_params, best_scale


## Recursive Forecasting

In [6]:
def tail_wet_streak(series: pd.Series) -> int:
    streak = 0
    for value in reversed(series.tolist()):
        if value >= RAIN_THRESHOLD:
            streak += 1
        else:
            break
    return streak


def tail_dry_streak(series: pd.Series) -> int:
    streak = 0
    for value in reversed(series.tolist()):
        if value < RAIN_THRESHOLD:
            streak += 1
        else:
            break
    return streak


def build_recursive_feature_row(history_df: pd.DataFrame, forecast_date: pd.Timestamp, feature_columns):
    rainfall_series = history_df["rainfall_mm"].reset_index(drop=True)
    rain_indicator_series = (rainfall_series >= RAIN_THRESHOLD).astype(int)
    row = {
        "month_sin": math.sin(2 * math.pi * forecast_date.month / 12),
        "month_cos": math.cos(2 * math.pi * forecast_date.month / 12),
        "day_of_year_sin": math.sin(2 * math.pi * forecast_date.dayofyear / 365.25),
        "day_of_year_cos": math.cos(2 * math.pi * forecast_date.dayofyear / 365.25),
        "quarter": int(((forecast_date.month - 1) // 3) + 1),
        "season_4": int(((forecast_date.month - 1) // 3) + 1),
        "is_rainy_season": int(forecast_date.month in [5, 6, 7, 8, 9, 10, 11]),
    }

    for lag in LAG_STEPS:
        row[f"lag_{lag}"] = float(rainfall_series.iloc[-lag])

    for window in ROLLING_WINDOWS:
        values = rainfall_series.tail(window)
        row[f"rolling_mean_{window}"] = float(values.mean())
        row[f"rolling_std_{window}"] = float(values.std()) if len(values) > 1 else 0.0
        row[f"rolling_sum_{window}"] = float(values.sum())
        row[f"rolling_max_{window}"] = float(values.max())
        row[f"rolling_min_{window}"] = float(values.min())

    for window in RAIN_COUNT_WINDOWS:
        row[f"rolling_rain_count_{window}"] = float(rain_indicator_series.tail(window).sum())

    rainy_dates = history_df.loc[history_df["rainfall_mm"] >= RAIN_THRESHOLD, "date"]
    if rainy_dates.empty:
        row["days_since_last_rain"] = float(len(history_df))
    else:
        row["days_since_last_rain"] = float((forecast_date - rainy_dates.iloc[-1]).days)

    row["wet_streak_prev"] = float(tail_wet_streak(rainfall_series))
    row["dry_streak_prev"] = float(tail_dry_streak(rainfall_series))

    feature_row = pd.DataFrame([row])
    return feature_row[feature_columns]


def fit_models(model_data: pd.DataFrame, feature_columns):
    train_df, val_df = time_validation_split(model_data)

    classifier_params, rain_threshold = tune_classifier(train_df, val_df, feature_columns)
    regressor_params, amount_scale = tune_regressor(train_df, val_df, feature_columns)

    classifier_model = build_classifier(classifier_params)
    classifier_model.fit(
        model_data[feature_columns],
        model_data["rain_occurrence"],
        sample_weight=classifier_weights(model_data),
    )

    rainy_rows = model_data[model_data["rain_occurrence"] == 1].copy()
    if rainy_rows.empty:
        regressor_model = None
    else:
        regressor_model = build_regressor(regressor_params)
        regressor_model.fit(
            rainy_rows[feature_columns],
            rainy_rows["target_sqrt_rainfall"],
            sample_weight=regressor_weights(rainy_rows),
        )

    return classifier_model, regressor_model, rain_threshold, amount_scale, classifier_params, regressor_params


def recursive_forecast(history_df: pd.DataFrame, classifier_model, regressor_model, feature_columns, rain_threshold: float, amount_scale: float) -> pd.DataFrame:
    history = history_df[["date", "rainfall_mm"]].copy().sort_values("date").reset_index(drop=True)
    start_date = history["date"].max() + pd.Timedelta(days=1)
    end_date = history["date"].max() + pd.DateOffset(months=12)
    forecast_dates = pd.date_range(start=start_date, end=end_date, freq="D")

    rows = []
    for forecast_date in forecast_dates:
        feature_row = build_recursive_feature_row(history, forecast_date, feature_columns)
        rain_probability = float(classifier_model.predict_proba(feature_row)[:, 1][0])
        rain_flag = int(rain_probability >= rain_threshold)

        if rain_flag == 0 or regressor_model is None:
            prediction_mm = 0.0
        else:
            pred_sqrt = float(regressor_model.predict(feature_row)[0])
            prediction_mm = max((max(pred_sqrt, 0.0) ** 2) * amount_scale, 0.0)

        rows.append(
            {
                "date": forecast_date,
                "year": forecast_date.year,
                "month": forecast_date.month,
                "day": forecast_date.day,
                "predicted_rainfall_mm": prediction_mm,
                "rain_probability": rain_probability,
                "prediction_stage": "recursive_daily",
            }
        )

        history.loc[len(history)] = {"date": forecast_date, "rainfall_mm": prediction_mm}

    return pd.DataFrame(rows)


## Province Loop and Export

In [7]:
def process_province_file(file_path: Path) -> pd.DataFrame:
    province_df = pd.read_csv(file_path)
    province_df["date"] = pd.to_datetime(province_df["date"])
    province_df["rainfall_mm"] = pd.to_numeric(province_df["rainfall_mm"], errors="coerce").fillna(0)
    province_df = province_df.sort_values("date").reset_index(drop=True)

    province_name = province_df["province_name"].dropna().iloc[0]
    model_data, feature_columns = build_training_frame(province_df)
    if model_data.empty:
        raise ValueError(f"Not enough history to create features for {province_name}")

    classifier_model, regressor_model, rain_threshold, amount_scale, classifier_params, regressor_params = fit_models(
        model_data,
        feature_columns,
    )
    forecast_df = recursive_forecast(
        history_df=province_df,
        classifier_model=classifier_model,
        regressor_model=regressor_model,
        feature_columns=feature_columns,
        rain_threshold=rain_threshold,
        amount_scale=amount_scale,
    )
    forecast_df.insert(0, "province_name", province_name)
    forecast_df["model_name"] = "xgboost_recent_weighted_daily"

    output_path = RESULT_DIR / file_path.name.replace("_rainfall_1981_2025.csv", "_forecast_next_12_months.csv")
    forecast_df.to_csv(output_path, index=False)

    print(
        f"Saved {output_path.name}: province={province_name}, train_rows={len(model_data):,}, "
        f"forecast_days={len(forecast_df):,}, rain_threshold={rain_threshold:.2f}, amount_scale={amount_scale:.2f}"
    )
    print(f"  classifier_params={classifier_params}")
    print(f"  regressor_params={regressor_params}")

    return forecast_df


all_forecasts = []
for province_file in province_files:
    all_forecasts.append(process_province_file(province_file))

combined_forecast_df = pd.concat(all_forecasts, ignore_index=True)
combined_path = RESULT_DIR / "all_provinces_forecast_next_12_months.csv"
combined_forecast_df.to_csv(combined_path, index=False)

print("\nCompleted forecasting for all provinces.")
print(f"Combined forecast saved to: {combined_path}")
combined_forecast_df.head()


Saved an_giang_forecast_next_12_months.csv: province=An Giang, train_rows=5,480, forecast_days=365, rain_threshold=0.34, amount_scale=1.09
  classifier_params={'n_estimators': 500, 'max_depth': 5, 'learning_rate': 0.04, 'min_child_weight': 4, 'gamma': 0.1, 'subsample': 0.88, 'colsample_bytree': 0.88, 'reg_alpha': 0.05, 'reg_lambda': 1.2}
  regressor_params={'n_estimators': 700, 'max_depth': 4, 'learning_rate': 0.03, 'min_child_weight': 5, 'gamma': 0.1, 'subsample': 0.82, 'colsample_bytree': 0.82, 'reg_alpha': 0.06, 'reg_lambda': 1.4}


Saved bac_lieu_forecast_next_12_months.csv: province=Bac Lieu, train_rows=5,480, forecast_days=365, rain_threshold=0.22, amount_scale=1.01
  classifier_params={'n_estimators': 700, 'max_depth': 4, 'learning_rate': 0.03, 'min_child_weight': 5, 'gamma': 0.15, 'subsample': 0.82, 'colsample_bytree': 0.82, 'reg_alpha': 0.08, 'reg_lambda': 1.5}
  regressor_params={'n_estimators': 500, 'max_depth': 5, 'learning_rate': 0.04, 'min_child_weight': 4, 'gamma': 0.05, 'subsample': 0.88, 'colsample_bytree': 0.88, 'reg_alpha': 0.04, 'reg_lambda': 1.2}


Saved ben_tre_forecast_next_12_months.csv: province=Ben Tre, train_rows=5,480, forecast_days=365, rain_threshold=0.22, amount_scale=1.07
  classifier_params={'n_estimators': 500, 'max_depth': 5, 'learning_rate': 0.04, 'min_child_weight': 4, 'gamma': 0.1, 'subsample': 0.88, 'colsample_bytree': 0.88, 'reg_alpha': 0.05, 'reg_lambda': 1.2}
  regressor_params={'n_estimators': 700, 'max_depth': 4, 'learning_rate': 0.03, 'min_child_weight': 5, 'gamma': 0.1, 'subsample': 0.82, 'colsample_bytree': 0.82, 'reg_alpha': 0.06, 'reg_lambda': 1.4}


Saved ca_mau_forecast_next_12_months.csv: province=Ca Mau, train_rows=5,480, forecast_days=365, rain_threshold=0.26, amount_scale=1.06
  classifier_params={'n_estimators': 700, 'max_depth': 4, 'learning_rate': 0.03, 'min_child_weight': 5, 'gamma': 0.15, 'subsample': 0.82, 'colsample_bytree': 0.82, 'reg_alpha': 0.08, 'reg_lambda': 1.5}
  regressor_params={'n_estimators': 700, 'max_depth': 4, 'learning_rate': 0.03, 'min_child_weight': 5, 'gamma': 0.1, 'subsample': 0.82, 'colsample_bytree': 0.82, 'reg_alpha': 0.06, 'reg_lambda': 1.4}


Saved can_tho_city_forecast_next_12_months.csv: province=Can Tho city, train_rows=5,480, forecast_days=365, rain_threshold=0.30, amount_scale=1.05
  classifier_params={'n_estimators': 700, 'max_depth': 4, 'learning_rate': 0.03, 'min_child_weight': 5, 'gamma': 0.15, 'subsample': 0.82, 'colsample_bytree': 0.82, 'reg_alpha': 0.08, 'reg_lambda': 1.5}
  regressor_params={'n_estimators': 700, 'max_depth': 4, 'learning_rate': 0.03, 'min_child_weight': 5, 'gamma': 0.1, 'subsample': 0.82, 'colsample_bytree': 0.82, 'reg_alpha': 0.06, 'reg_lambda': 1.4}


...

Saved soc_trang_forecast_next_12_months.csv: province=Soc Trang, train_rows=5,480, forecast_days=365, rain_threshold=0.30, amount_scale=1.02
  classifier_params={'n_estimators': 700, 'max_depth': 4, 'learning_rate': 0.03, 'min_child_weight': 5, 'gamma': 0.15, 'subsample': 0.82, 'colsample_bytree': 0.82, 'reg_alpha': 0.08, 'reg_lambda': 1.5}
  regressor_params={'n_estimators': 500, 'max_depth': 5, 'learning_rate': 0.04, 'min_child_weight': 4, 'gamma': 0.05, 'subsample': 0.88, 'colsample_bytree': 0.88, 'reg_alpha': 0.04, 'reg_lambda': 1.2}


Saved tien_giang_forecast_next_12_months.csv: province=Tien Giang, train_rows=5,480, forecast_days=365, rain_threshold=0.18, amount_scale=1.10
  classifier_params={'n_estimators': 700, 'max_depth': 4, 'learning_rate': 0.03, 'min_child_weight': 5, 'gamma': 0.15, 'subsample': 0.82, 'colsample_bytree': 0.82, 'reg_alpha': 0.08, 'reg_lambda': 1.5}
  regressor_params={'n_estimators': 500, 'max_depth': 5, 'learning_rate': 0.04, 'min_child_weight': 4, 'gamma': 0.05, 'subsample': 0.88, 'colsample_bytree': 0.88, 'reg_alpha': 0.04, 'reg_lambda': 1.2}


Saved tra_vinh_forecast_next_12_months.csv: province=Tra Vinh, train_rows=5,480, forecast_days=365, rain_threshold=0.34, amount_scale=0.96
  classifier_params={'n_estimators': 500, 'max_depth': 5, 'learning_rate': 0.04, 'min_child_weight': 4, 'gamma': 0.1, 'subsample': 0.88, 'colsample_bytree': 0.88, 'reg_alpha': 0.05, 'reg_lambda': 1.2}
  regressor_params={'n_estimators': 700, 'max_depth': 4, 'learning_rate': 0.03, 'min_child_weight': 5, 'gamma': 0.1, 'subsample': 0.82, 'colsample_bytree': 0.82, 'reg_alpha': 0.06, 'reg_lambda': 1.4}


Saved vinh_long_forecast_next_12_months.csv: province=Vinh Long, train_rows=5,480, forecast_days=365, rain_threshold=0.18, amount_scale=0.98
  classifier_params={'n_estimators': 700, 'max_depth': 4, 'learning_rate': 0.03, 'min_child_weight': 5, 'gamma': 0.15, 'subsample': 0.82, 'colsample_bytree': 0.82, 'reg_alpha': 0.08, 'reg_lambda': 1.5}
  regressor_params={'n_estimators': 700, 'max_depth': 4, 'learning_rate': 0.03, 'min_child_weight': 5, 'gamma': 0.1, 'subsample': 0.82, 'colsample_bytree': 0.82, 'reg_alpha': 0.06, 'reg_lambda': 1.4}

Completed forecasting for all provinces.
Combined forecast saved to: C:\Users\Administrator\Desktop\DataViz_FinalPrj\COMP4010-Project-2\modeling\result\all_provinces_forecast_next_12_months.csv


,province_name,date,year,month,day,predicted_rainfall_mm,rain_probability,prediction_stage,model_name
0,An Giang,2026-01-01,2026,1,1,0.0,0.054688,recursive_daily,xgboost_recent_weighted_daily
1,An Giang,2026-01-02,2026,1,2,0.0,0.031511,recursive_daily,xgboost_recent_weighted_daily
2,An Giang,2026-01-03,2026,1,3,0.0,0.137786,recursive_daily,xgboost_recent_weighted_daily
3,An Giang,2026-01-04,2026,1,4,0.0,0.038366,recursive_daily,xgboost_recent_weighted_daily
4,An Giang,2026-01-05,2026,1,5,0.0,0.039375,recursive_daily,xgboost_recent_weighted_daily
